# LegalQA Task 2 — Canonical Reproducible Dual-T4 Kaggle Pipeline (Stack A)
Production LegalQA training, validation, and inference pipeline on Kaggle Dual NVIDIA T4 GPUs.
- **Stack A Production Architecture**: Exact/Similar QA Memory -> BM25S (mmap) + Dense DEk21 v2 (FP16 GPU top-K) -> RRF Fusion -> Task-Tuned BGE Reranker -> Structured Evidence Packer -> Qwen2.5-3B-Instruct (4-bit QLoRA) -> Candidate Ensemble & Selector Guardrail.
- **Hardware Layout**: Dual NVIDIA T4 (GPU 0: Qwen Generator | GPU 1: DEk21 Dense + BGE Reranker | CPU: BM25S + QA Memory + Selector).

In [ ]:
# Cell 1 — Execution Profile Configuration
SEED = 42
# Execution Profiles:
#   'smoke_only': Fast code & hardware verification (~30 optimizer steps, 5 test queries)
#   'screen_fold0': Excludes fold 0 from training and evaluates on held-out fold 0
#   'final_train_and_submit': Trains on ALL allowed data, validates checkpoints, and creates submission
#   'reuse_final_checkpoints_and_submit': Uses existing verified final checkpoints and creates submission
EXECUTION_PROFILE = "final_train_and_submit"
ALLOW_INDEX_REBUILD = False  # Strict production policy: fail loudly if precomputed indexes are missing

if EXECUTION_PROFILE == "smoke_only":
    RUN_RERANKER_TRAINING = True
    RUN_GENERATOR_TRAINING = True
    RUN_DEV_EVALUATION = True
    RUN_PUBLIC_INFERENCE = False
    REUSE_EXISTING_CHECKPOINTS = False
    TRAIN_VAL_FOLD = 0
    MAX_SMOKE_STEPS = 30
elif EXECUTION_PROFILE == "screen_fold0":
    RUN_RERANKER_TRAINING = True
    RUN_GENERATOR_TRAINING = True
    RUN_DEV_EVALUATION = True
    RUN_PUBLIC_INFERENCE = False
    REUSE_EXISTING_CHECKPOINTS = False
    TRAIN_VAL_FOLD = 0
    MAX_SMOKE_STEPS = None
elif EXECUTION_PROFILE == "final_train_and_submit":
    RUN_RERANKER_TRAINING = True
    RUN_GENERATOR_TRAINING = True
    RUN_DEV_EVALUATION = False
    RUN_PUBLIC_INFERENCE = True
    REUSE_EXISTING_CHECKPOINTS = False
    TRAIN_VAL_FOLD = None  # Train on ALL allowed Task 2 data
    MAX_SMOKE_STEPS = None
elif EXECUTION_PROFILE == "reuse_final_checkpoints_and_submit":
    RUN_RERANKER_TRAINING = False
    RUN_GENERATOR_TRAINING = False
    RUN_DEV_EVALUATION = False
    RUN_PUBLIC_INFERENCE = True
    REUSE_EXISTING_CHECKPOINTS = True
    TRAIN_VAL_FOLD = None
    MAX_SMOKE_STEPS = None
else:
    raise ValueError(f"Unknown EXECUTION_PROFILE: {EXECUTION_PROFILE}")

print("===========================================================")
print(f"EXECUTION PROFILE:          {EXECUTION_PROFILE}")
print(f"RUN RERANKER TRAINING:      {RUN_RERANKER_TRAINING}")
print(f"RUN GENERATOR TRAINING:     {RUN_GENERATOR_TRAINING}")
print(f"TRAINING EXCLUDED FOLD:     {TRAIN_VAL_FOLD} (None = All Data)")
print(f"RUN DEV EVALUATION:         {RUN_DEV_EVALUATION}")
print(f"RUN PUBLIC INFERENCE:       {RUN_PUBLIC_INFERENCE}")
print("===========================================================")

In [ ]:
# Cell 2 — Environment, Secrets & Hardware Verification
import os, sys, gc, glob, json, zipfile, re, math, time, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Set deterministic seeds
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Securely retrieve HuggingFace Token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("HF_TOKEN securely loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN detected from environment.")
    else:
        print("Notice: HF_TOKEN secret not present; using public weights.")

# Dual-GPU Device Allocation
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
if EXECUTION_PROFILE in ("final_train_and_submit", "reuse_final_checkpoints_and_submit"):
    if gpu_count < 2:
        print(f"Warning: Detected {gpu_count} GPU(s). Dual T4 is recommended for full competition execution.")

if gpu_count >= 2:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:1"
elif gpu_count == 1:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:0"
else:
    GEN_DEVICE = "cpu"
    RETRIEVAL_DEVICE = "cpu"

print(f"CUDA GPUs Detected: {gpu_count}")
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.1f} GB | Compute: sm_{p.major}{p.minor}")
print(f"Hardware Allocation -> Generator: {GEN_DEVICE} | Retrieval/Reranker: {RETRIEVAL_DEVICE}")

In [ ]:
# Cell 3 — Deterministic Path Resolution for Runtime Code & Data
# 1. Resolve Code Root containing src/ and scripts/
code_candidates = glob.glob("/kaggle/input/**/code/LegalQA", recursive=True) + [".", "/kaggle/working", "/kaggle/working/LegalQA"]
resolved_code_root = None
for cand in code_candidates:
    if os.path.exists(os.path.join(cand, "src")) and os.path.exists(os.path.join(cand, "scripts")):
        resolved_code_root = os.path.abspath(cand)
        if resolved_code_root not in sys.path:
            sys.path.insert(0, resolved_code_root)
        print(f"Packaged Code Root resolved: {resolved_code_root}")
        break

assert resolved_code_root is not None, "Failed to resolve packaged code directory containing src/ and scripts/!"

# 2. Resolve Data Artifacts
qa_files = glob.glob("/kaggle/input/**/qa_unique.parquet", recursive=True) or glob.glob("artifacts/**/qa_unique.parquet", recursive=True)
chunks_files = glob.glob("/kaggle/input/**/legal_chunks.parquet", recursive=True) or glob.glob("artifacts/**/legal_chunks.parquet", recursive=True)
known_files = glob.glob("/kaggle/input/**/known_qa.json", recursive=True) or glob.glob("artifacts/**/known_qa.json", recursive=True)
test_files = glob.glob("/kaggle/input/**/public-official.json", recursive=True) or glob.glob("artifacts/**/public-official.json", recursive=True)

assert qa_files, "qa_unique.parquet not found!"
assert chunks_files, "legal_chunks.parquet not found!"
assert known_files, "known_qa.json not found!"
assert test_files, "public-official.json not found!"

QA_PATH = qa_files[0]
CHUNKS_PATH = chunks_files[0]
KNOWN_QA_PATH = known_files[0]
TEST_PATH = test_files[0]
DATA_DIR = os.path.dirname(QA_PATH)

# 3. Resolve Index Directories
bm25_dirs = [d for d in glob.glob("/kaggle/input/**/bm25*", recursive=True) if os.path.isdir(d)] or ["artifacts/task2/indexes/bm25"]
dek21_dirs = [d for d in glob.glob("/kaggle/input/**/dek21*", recursive=True) if os.path.isdir(d)] or ["artifacts/task2/indexes/dek21"]
BM25_DIR = bm25_dirs[0] if bm25_dirs and os.path.exists(bm25_dirs[0]) else "artifacts/task2/indexes/bm25"
DEK21_DIR = dek21_dirs[0] if dek21_dirs and os.path.exists(dek21_dirs[0]) else "artifacts/task2/indexes/dek21"

# 4. Resolve Mounted Qwen Model
qwen_configs = glob.glob("/kaggle/input/**/qwen*3b*/**/config.json", recursive=True) + glob.glob("/kaggle/input/**/3b-instruct*/**/config.json", recursive=True)
if qwen_configs:
    MODEL_PATH = os.path.dirname(qwen_configs[0])
else:
    MODEL_PATH = "Qwen/Qwen2.5-3B-Instruct"

print(f"Data Directory: {DATA_DIR}")
print(f"BM25 Index:     {BM25_DIR}")
print(f"Dense Index:    {DEK21_DIR}")
print(f"Qwen Base Path: {MODEL_PATH}")

In [ ]:
# Cell 4 — Install Missing Dependencies & Print Version Tuple
req_path = os.path.join(resolved_code_root, "requirements-kaggle.txt")
if os.path.exists(req_path):
    print(f"Validating dependencies from {req_path}...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", req_path])
        print("Dependencies successfully verified!")
    except Exception as e:
        print(f"Warning during pip install: {e}", file=sys.stderr)

import importlib.metadata as md
for pkg in ["torch", "transformers", "peft", "trl", "bitsandbytes", "sentence-transformers", "bm25s", "pyvi"]:
    try:
        print(f" - {pkg:25s}: {md.version(pkg)}")
    except Exception:
        print(f" - {pkg:25s}: not installed")

In [ ]:
# Cell 5 — Strict Production Preflight Diagnostics
from scripts.preflight_kaggle import run_preflight_checks

preflight_res = run_preflight_checks(
    pipeline_config_path="configs/pipeline.yaml" if os.path.exists("configs/pipeline.yaml") else os.path.join(resolved_code_root, "configs/pipeline.yaml"),
    models_config_path="configs/models.yaml" if os.path.exists("configs/models.yaml") else os.path.join(resolved_code_root, "configs/models.yaml"),
    require_cuda=torch.cuda.is_available(),
    expected_gpu_count=2 if torch.cuda.device_count() >= 2 else 1,
    check_dataset_files=True,
    data_dir=DATA_DIR,
    bm25_dir=BM25_DIR,
    dek21_dir=DEK21_DIR,
    public_path=TEST_PATH,
    stack="stack_a",
    require_training_files=RUN_RERANKER_TRAINING,
)

if not preflight_res["passed"]:
    print("Preflight validation failed:", preflight_res["errors"])
    if not ALLOW_INDEX_REBUILD:
        raise RuntimeError(f"PREFLIGHT FAILED: {preflight_res['errors']}")
else:
    print("Production preflight completely PASSED!")

In [ ]:
# Cell 6 — Load QA & Index Metadata without Duplication
from src.task2.qa_memory import QAMemory
from src.common.bm25 import BM25Retriever

print("Loading verified QA Memory...")
memory = QAMemory.load(KNOWN_QA_PATH, QA_PATH)
print(f"Loaded QA Memory: {len(memory.id_to_answer):,} IDs | {len(memory.question_to_answer):,} unique questions.")

print("Loading BM25 Index (mmap)... ")
if os.path.exists(os.path.join(BM25_DIR, "bm25_manifest.json")) or os.path.exists(os.path.join(BM25_DIR, "bm25s_index")):
    bm25 = BM25Retriever.load(BM25_DIR, corpus_path=CHUNKS_PATH, fail_on_missing_index=not ALLOW_INDEX_REBUILD)
else:
    if not ALLOW_INDEX_REBUILD:
        raise FileNotFoundError(f"FINAL_PIPELINE_ERROR: BM25 index missing at {BM25_DIR}")
    print("Rebuilding BM25 index from legal chunks...")
    df_chunks = pd.read_parquet(CHUNKS_PATH)
    bm25 = BM25Retriever()
    bm25.fit(df_chunks.to_dict("records"))
print(f"BM25 Ready: {bm25.corpus_size:,} chunks indexed.")

In [ ]:
# Cell 7 — Task-Tuned Reranker Fine-Tuning
RERANKER_CHECKPOINT = "BAAI/bge-reranker-v2-m3"

if RUN_RERANKER_TRAINING:
    from src.task2.training.train_reranker import train_bge_reranker
    pairs_path = os.path.join(DATA_DIR, "reranker_training_pairs.parquet")
    reranker_out = "/kaggle/working/checkpoints/reranker/best"
    print(f"Starting Reranker fine-tuning on {RETRIEVAL_DEVICE} (val_fold={TRAIN_VAL_FOLD}, max_steps={MAX_SMOKE_STEPS})...")
    res_rerank = train_bge_reranker(
        pairs_path=pairs_path,
        output_dir=reranker_out,
        model_name="BAAI/bge-reranker-v2-m3",
        epochs=1,
        batch_size=2,
        grad_accum=4,
        lr=2e-5,
        val_fold=TRAIN_VAL_FOLD,
        max_steps=MAX_SMOKE_STEPS,
        device=RETRIEVAL_DEVICE,
        fail_on_error=True,
    )
    if res_rerank.get("status") != "completed":
        raise RuntimeError(f"Reranker training requested but failed: {res_rerank}")
    RERANKER_CHECKPOINT = reranker_out
    print(f"Reranker training complete! Best checkpoint saved to {RERANKER_CHECKPOINT}")
elif REUSE_EXISTING_CHECKPOINTS:
    ckpt_candidates = glob.glob("/kaggle/input/**/checkpoints/reranker/best", recursive=True)
    if ckpt_candidates:
        RERANKER_CHECKPOINT = ckpt_candidates[0]
        print(f"Reusing pre-trained reranker checkpoint: {RERANKER_CHECKPOINT}")
    else:
        print(f"No pre-trained reranker checkpoint found. Using base: {RERANKER_CHECKPOINT}")
else:
    print(f"Using pretrained base reranker: {RERANKER_CHECKPOINT}")

In [ ]:
# Cell 8 — Qwen2.5-3B QLoRA SFT Fine-Tuning & Reload Smoke Test
ADAPTER_PATH = None

if RUN_GENERATOR_TRAINING:
    from src.task2.training.train_generator import run_qlora_training
    labels_path = os.path.join(DATA_DIR, "retrieval_labels.parquet")
    qlora_out = "/kaggle/working/checkpoints/generator/hf_adapter"
    print(f"Starting QLoRA fine-tuning on {GEN_DEVICE} (val_fold={TRAIN_VAL_FOLD}, max_steps={MAX_SMOKE_STEPS})...")
    res_qlora = run_qlora_training(
        model_name=MODEL_PATH,
        qa_path=QA_PATH,
        labels_path=labels_path,
        chunks_path=CHUNKS_PATH,
        output_dir=qlora_out,
        epochs=1,
        batch_size=1,
        grad_accum=8,
        lr=1e-4,
        max_seq_len=2048,
        val_fold=TRAIN_VAL_FOLD,
        max_steps=MAX_SMOKE_STEPS,
        device=GEN_DEVICE,
        fail_on_error=True,
    )
    if res_qlora.get("status") != "completed":
        raise RuntimeError(f"QLoRA training requested but failed: {res_qlora}")
    ADAPTER_PATH = qlora_out
    print(f"QLoRA training complete! Adapter verified and saved to {ADAPTER_PATH}")
elif REUSE_EXISTING_CHECKPOINTS:
    ad_candidates = glob.glob("/kaggle/input/**/checkpoints/generator/hf_adapter", recursive=True)
    if ad_candidates:
        ADAPTER_PATH = ad_candidates[0]
        print(f"Reusing pre-trained QLoRA adapter: {ADAPTER_PATH}")
    else:
        print("No pre-trained adapter found. Using base generator.")
else:
    print("QLoRA training skipped. Using base generator.")

In [ ]:
# Cell 9 — Exact Parameter Audit Including Adapter
from scripts.audit_parameters import audit_parameter_budget

models_cfg = "configs/models.yaml" if os.path.exists("configs/models.yaml") else os.path.join(resolved_code_root, "configs/models.yaml")
adapter_manifest = os.path.join(ADAPTER_PATH, "training_manifest.json") if ADAPTER_PATH else None

audit_res = audit_parameter_budget(models_cfg, stack="stack_a", adapter_manifest_path=adapter_manifest)
print("=== FINAL STACK PARAMETER AUDIT ===")
for k, v in audit_res["breakdown"].items():
    print(f" - {k:45s}: {v:,} parameters")
print(f"TOTAL LEARNED PARAMETERS: {audit_res['total_learned_parameters']:,}")
print(f"OFFICIAL LIMIT:            {audit_res['limit']:,} (strict exclusive)")
print(f"REMAINING SAFE MARGIN:     {audit_res['margin']:,}")
print(f"COMPLIANCE STATUS:         {'COMPLIANT' if audit_res['is_compliant'] else 'NON-COMPLIANT'}")

if not audit_res["is_compliant"]:
    raise RuntimeError(f"PARAMETER BUDGET EXCEEDED: {audit_res['total_learned_parameters']:,} >= {audit_res['limit']:,}")

In [ ]:
# Cell 10 — Post-Training Sanity Evaluation on Held-Out Validation Fold
if RUN_DEV_EVALUATION:
    from src.task2.evaluation import evaluate_checkpoint
    eval_fold = TRAIN_VAL_FOLD if TRAIN_VAL_FOLD is not None else 0
    eval_sample_size = 5 if EXECUTION_PROFILE == "smoke_only" else 50
    print(f"Running post-training evaluation on held-out fold {eval_fold} ({eval_sample_size} samples)...")
    eval_res = evaluate_checkpoint(
        qa_path=QA_PATH,
        fold_path=os.path.join(DATA_DIR, "fold_assignments.parquet"),
        chunks_path=CHUNKS_PATH,
        held_out_fold=eval_fold,
        bm25_dir=BM25_DIR,
        dense_dir=DEK21_DIR,
        dense_model="CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2",
        reranker_checkpoint=RERANKER_CHECKPOINT,
        generator_model=MODEL_PATH,
        adapter_path=ADAPTER_PATH,
        sample_size=eval_sample_size,
        eval_output_dir="/kaggle/working/evaluations",
        gen_device=GEN_DEVICE,
        retrieval_device=RETRIEVAL_DEVICE,
        fail_on_fallback=False if EXECUTION_PROFILE == "smoke_only" else True,
    )
    print(f"Evaluation Complete -> Selected METEOR: {eval_res['selected_meteor']:.4f} | Oracle: {eval_res['oracle_meteor']:.4f}")
else:
    print("Dev evaluation skipped.")

In [ ]:
# Cell 11 — Load Final Dual-T4 Inference Pipeline
from src.task2.predict import LegalQAPipeline
from src.common.dense import DenseRetriever
from src.common.reranker import BGEReranker
from src.task2.evidence_packer import EvidencePacker
from src.task2.generator import QwenGenerator
from src.task2.selector import CandidateSelector

# Clean memory from training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Initializing final Dual-T4 pipeline...")
# 1. Dense DEk21 on GPU 1 (mmap FP16)
dense = DenseRetriever.load_index(
    DEK21_DIR,
    corpus_path=CHUNKS_PATH,
    device=RETRIEVAL_DEVICE,
    expected_model_name="CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2",
    final_mode=True if EXECUTION_PROFILE == "final_train_and_submit" else False,
)

# 2. Reranker on GPU 1
print(f"Loading Reranker from {RERANKER_CHECKPOINT} on {RETRIEVAL_DEVICE}...")
reranker = BGEReranker(model_name=RERANKER_CHECKPOINT, device=RETRIEVAL_DEVICE)

# 3. Evidence Packer on CPU
packer = EvidencePacker(bm25.corpus)

# 4. Qwen Generator on GPU 0
print(f"Loading Qwen Generator on {GEN_DEVICE}...")
generator = QwenGenerator.load(
    model_path=MODEL_PATH,
    adapter_path=ADAPTER_PATH,
    device=GEN_DEVICE,
    runtime="torch" if torch.cuda.is_available() else "fallback",
    fail_on_fallback=True if torch.cuda.is_available() and EXECUTION_PROFILE == "final_train_and_submit" else False,
    final_mode=True if EXECUTION_PROFILE == "final_train_and_submit" else False,
)

# 5. Selector on CPU with Best Fixed Baseline Guardrail
selector = CandidateSelector(policy="fixed_baseline", best_fixed_candidate="stitched_extract")

pipeline = LegalQAPipeline(memory, bm25, dense, reranker, packer, generator, selector)
print("Dual-T4 Production Pipeline loaded successfully!")

In [ ]:
# Cell 12 — Load Public Test Set & Execute True Batched Inference
with open(TEST_PATH, "r", encoding="utf-8") as f:
    public_test = json.load(f)

print(f"Loaded {len(public_test)} public test questions from {TEST_PATH}.")

items_to_predict = [{"id": str(qid), "question": str(item.get("question", "")).strip()} for qid, item in public_test.items()]

if RUN_PUBLIC_INFERENCE:
    start_time = time.time()
    print(f"Executing true batched inference on {len(items_to_predict)} questions...")
    batch_size_gen = 2 if torch.cuda.is_available() else 1
    submission = pipeline.predict_batch(
        items=items_to_predict,
        max_new_tokens=384,
        retrieval_batch_size=32,
        generation_batch_size=batch_size_gen,
    )
    elapsed = time.time() - start_time
    print(f"Inference completed in {elapsed:.1f}s ({len(submission)} predictions generated).")
else:
    print("Public inference skipped by configuration.")
    submission = {}

In [ ]:
# Cell 13 — Strict Submission Verification & Diagnostics
if RUN_PUBLIC_INFERENCE:
    print("=== Strict Submission Verification ===")
    assert len(submission) == 1000, f"Submission count mismatch! Expected 1000, got {len(submission)}"

    test_keys = set(public_test.keys())
    sub_keys = set(submission.keys())
    assert test_keys == sub_keys, f"Submission keys mismatch! Diff: {test_keys ^ sub_keys}"

    for qid, val in submission.items():
        ans = val.get("answer", "")
        assert isinstance(ans, str) and len(ans.strip()) > 0, f"Empty answer for ID {qid}!"
        assert "[DOCUMENT]" not in ans and "[ARTICLE]" not in ans, f"Internal tag found in ID {qid}!"

    lengths = [len(v["answer"].split()) for v in submission.values()]
    print(f"Total Queries:        {len(submission):,}")
    print(f"Mean Word Count:      {np.mean(lengths):.1f} words")
    print(f"Median Word Count:    {np.median(lengths):.1f} words")
    print(f"P90 Word Count:       {np.percentile(lengths, 90):.1f} words")
    print(f"Min / Max Length:     {np.min(lengths)} / {np.max(lengths)} words")
    print("All submission verification gates PASSED!")

In [ ]:
# Cell 14 — Save Submission Archive & Provenance Manifest
if RUN_PUBLIC_INFERENCE:
    out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "artifacts/task2/submissions"
    os.makedirs(out_dir, exist_ok=True)

    out_json = os.path.join(out_dir, "submission.json")
    out_zip = os.path.join(out_dir, "submission.json.zip")
    manifest_path = os.path.join(out_dir, "run_manifest.json")

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_json, arcname="submission.json")

    run_manifest = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        "execution_profile": EXECUTION_PROFILE,
        "num_queries": len(submission),
        "stack": "stack_a",
        "reranker_checkpoint": RERANKER_CHECKPOINT,
        "adapter_path": ADAPTER_PATH,
        "mean_word_count": float(np.mean(lengths)),
        "learned_parameters": audit_res["total_learned_parameters"],
    }
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(run_manifest, f, indent=2)

    print(f"Created: {out_json} ({os.path.getsize(out_json)/1024:.1f} KB)")
    print(f"Created: {out_zip} ({os.path.getsize(out_zip)/1024:.1f} KB)")
    print(f"Created: {manifest_path}")
    print("\nSUCCESS: Kaggle Stack A pipeline finished and validated!")